# NBA Schedule — Combined Notebook

Combines:
- **Project II Q1–Q3 (Gurobi)**: schedule statistics, feasibility IP, and time-zone smoothness (with relaxed structural analysis).
- **Arena geocoding & distance matrix** (from BBR scraper Part 2).
- **Minimax travel IP (Part 3)** and **per-team travel from `games.csv` (Part 4)** (from team-schedule analysis).

**Variable conventions** (kept consistent across sections):
- `games`  — schedule DataFrame from `games.csv` (with `Date`, `DateStr`, `Date_only`, `Home`, `Visitor`, `Arena`).
- `teams`  — sorted list of team names.
- `dates`  — sorted list of `DateStr` (ISO strings); `n = len(dates)`.
- `summary` — Q1 (a)(b)(c)(d) per team.
- `home_date_set`, `away_date_set`, `h_count`, `a_count`, `valid_trips` — sparse Q2 inputs.
- `team_tz` — team → time-zone index (ET=0, CT=1, MT=2, PT=3).
- `arenas`, `arena_coords`, `dist_df` — Part 2 arena set + distance matrix (km).
- `D`, `nD`, `nT`, `team_index`, `D_index`, `alpha`, `beta`, `b`, `c` — Part 3 IP parameters (derived from `summary`).


In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from IPython.display import display


def find_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "games.csv").exists():
            return p
    raise FileNotFoundError("games.csv not found")


ROOT    = find_root()
OUT_DIR = ROOT / "output"
OUT_DIR.mkdir(exist_ok=True)


# Helper used by Part 3 to export per-team matchup workbooks.
def _sanitize_excel_sheet_name(name: str, used: set[str]) -> str:
    s = re.sub(r"[:\\/*?\[\]]", " ", str(name)).strip() or "Team"
    s = s[:31]
    base = s
    k = 2
    while s in used:
        suffix = f" ({k})"
        s = (base[: 31 - len(suffix)] + suffix) if len(base) + len(suffix) > 31 else base + suffix
        k += 1
    used.add(s)
    return s


def matchup_table_for_team(i: str, schedule: pd.DataFrame) -> pd.DataFrame:
    home = schedule.loc[schedule["Home"] == i, ["Date_only", "Visitor"]].rename(columns={"Visitor": "j"})
    away = schedule.loc[schedule["Visitor"] == i, ["Date_only", "Home"]].rename(columns={"Home": "j"})
    all_j = sorted(set(home["j"]).union(away["j"]))
    rows = []
    for j in all_j:
        hj = home.loc[home["j"] == j, "Date_only"]
        aj = away.loc[away["j"] == j, "Date_only"]
        home_dates = sorted(hj.dt.normalize().unique()) if len(hj) else []
        away_dates = sorted(aj.dt.normalize().unique()) if len(aj) else []
        rows.append({
            "Team j":               j,
            "Games vs j at home":   int(len(hj)),
            "Home game dates vs j": "; ".join(str(pd.Timestamp(d).date()) for d in home_dates),
            "Games vs j away":      int(len(aj)),
            "Away game dates at j": "; ".join(str(pd.Timestamp(d).date()) for d in away_dates),
        })
    return pd.DataFrame(rows)


def write_team_matchups_excel(schedule: pd.DataFrame, path: Path) -> None:
    team_list = sorted(set(schedule["Home"]).union(schedule["Visitor"]))
    used: set[str] = set()
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for i in team_list:
            sheet = _sanitize_excel_sheet_name(i, used)
            matchup_table_for_team(i, schedule=schedule).to_excel(writer, sheet_name=sheet, index=False)


## Load schedule

In [ ]:
games = pd.read_csv(ROOT / "games.csv")

# Pandas may rename duplicate "PTS" columns to PTS / PTS.1 — normalise.
games = games.rename(columns={"PTS.1": "Home_PTS"})
if "PTS" in games.columns and "Visitor_PTS" not in games.columns:
    games = games.rename(columns={"PTS": "Visitor_PTS"})

games["Date"]      = pd.to_datetime(games["Date"], format="mixed", utc=False)
games["DateStr"]   = games["Date"].dt.strftime("%Y-%m-%d")
games["Date_only"] = games["Date"].dt.normalize()
if "Arena" in games.columns:
    games["Arena"] = games["Arena"].astype(str).str.strip()

teams = sorted(set(games["Home"]) | set(games["Visitor"]))
dates = sorted(games["DateStr"].unique())
n     = len(dates)

print(f"Games: {len(games)}  |  Teams: {len(teams)}  |  Dates: {n}")
display(games[["DateStr", "Home", "Visitor", "Arena"]].head())


## Question 1 — Schedule statistics (a)(b)(c)(d)

In [ ]:
def q1_stats(schedule, team_list):
    """
    For every team return:
      (a) home_dates  : dates the team played at home
      (b) home_counts : {opponent: number of home games vs opponent}
      (c) away_counts : {opponent: number of away games at opponent's arena}
      (d) away_dates  : dates the team played away
    """
    stats = {}
    for t in team_list:
        home_rows = schedule[schedule["Home"]    == t]
        away_rows = schedule[schedule["Visitor"] == t]
        stats[t] = {
            "home_dates":  sorted(home_rows["DateStr"].tolist()),
            "home_counts": {
                j: int(((schedule["Home"] == t) & (schedule["Visitor"] == j)).sum())
                for j in team_list if j != t
            },
            "away_counts": {
                j: int(((schedule["Visitor"] == t) & (schedule["Home"] == j)).sum())
                for j in team_list if j != t
            },
            "away_dates":  sorted(away_rows["DateStr"].tolist()),
        }
    return stats


summary = q1_stats(games, teams)

for team in teams:
    s = summary[team]
    print(f"=== {team} ===")
    print(f"  (a) Home dates : {s['home_dates']}")
    print(f"  (b) Home counts: {s['home_counts']}")
    print(f"  (c) Away counts: {s['away_counts']}")
    print(f"  (d) Away dates : {s['away_dates']}")
    print()


In [ ]:
# Per-team matchup Excel (same content as Q1, in workbook form)
out_q1_xlsx = ROOT / "team_matchups_by_opponent.xlsx"
write_team_matchups_excel(games, out_q1_xlsx)
print(f"Wrote {out_q1_xlsx.resolve()}")


## Question 2 — Feasibility IP model (e)(f)(g)(h)

**Decision variable**: $x_{d,i,j} \in \{0,1\}$ — equals 1 if team $i$ hosts team $j$ on date $d$.

**Constraints (no objective — pure feasibility)**:
- **(e)** $\sum_j x_{d,i,j} = 1$ for every home date $d$ of $i$
- **(f)** $\sum_j x_{d,j,i} = 1$ for every away date $d$ of $i$
- **(g)** $\sum_d x_{d,i,j} = h_{ij}$ — season home matchup count matches original
- **(h)** $\sum_d x_{d,j,i} = a_{ij}$ — season away matchup count matches original

**Variable sparsity**: `x[d,i,j]` is only created when `d` is a home date of `i` **and** an away date of `j`.


In [ ]:
# Pre-compute right-hand sides (derived from Q1)
home_date_set = {t: set(summary[t]["home_dates"]) for t in teams}
away_date_set = {t: set(summary[t]["away_dates"]) for t in teams}
h_count       = {(i, j): summary[i]["home_counts"][j] for i in teams for j in teams if i != j}
a_count       = {(i, j): summary[i]["away_counts"][j] for i in teams for j in teams if i != j}

# Sparse variable index set: x[d,i,j] only exists when d is a home date of i AND an away date of j.
valid_trips = [
    (d, i, j)
    for d in dates
    for i in teams
    for j in teams
    if i != j
    and d in home_date_set[i]
    and d in away_date_set[j]
]
print(f"Sparse variable count : {len(valid_trips)}")
print(f"Dense variable count  : {len(dates) * len(teams)**2}  (exceeds free-license limit)")


In [ ]:
def build_q2_gurobi(home_date_set, away_date_set, h_count, a_count, valid_trips):
    """Build the Q2 feasibility model. All inputs passed explicitly — no hidden global state."""
    m = gp.Model("NBA_Q2")
    m.setParam("OutputFlag", 0)

    x = m.addVars(valid_trips, vtype=GRB.BINARY, name="x")

    # (e) On each home date d, team i must host exactly one opponent
    for i in teams:
        for d in home_date_set[i]:
            m.addConstr(
                gp.quicksum(x[d, i, j] for j in teams
                            if j != i and d in away_date_set[j]) == 1,
                name=f"e_{d}_{i}",
            )

    # (f) On each away date d, team i must visit exactly one opponent's arena
    for i in teams:
        for d in away_date_set[i]:
            m.addConstr(
                gp.quicksum(x[d, j, i] for j in teams
                            if j != i and d in home_date_set[j]) == 1,
                name=f"f_{d}_{i}",
            )

    # (g) Season home games of i vs j
    for i in teams:
        for j in teams:
            if i == j:
                continue
            m.addConstr(
                gp.quicksum(x[d, i, j] for d in dates
                            if d in home_date_set[i] and d in away_date_set[j])
                == h_count[(i, j)],
                name=f"g_{i}_{j}",
            )

    # (h) Season away games of i at j's arena
    for i in teams:
        for j in teams:
            if i == j:
                continue
            m.addConstr(
                gp.quicksum(x[d, j, i] for d in dates
                            if d in home_date_set[j] and d in away_date_set[i])
                == a_count[(i, j)],
                name=f"h_{i}_{j}",
            )

    m.setObjective(0, GRB.MINIMIZE)
    return m, x


q2_model, q2_x = build_q2_gurobi(home_date_set, away_date_set, h_count, a_count, valid_trips)
print(f"Variables  : {q2_model.NumVars}")
print(f"Constraints: {q2_model.NumConstrs}")


In [ ]:
q2_model.optimize()


def decode_schedule(xv, valid_trips):
    """Decode a solved Gurobi model into a schedule DataFrame."""
    rows = [
        {"Date": d, "DateStr": d, "Home": i, "Visitor": j}
        for (d, i, j) in valid_trips
        if xv[d, i, j].X > 0.5
    ]
    return pd.DataFrame(rows).sort_values(["Date", "Home"]).reset_index(drop=True)


if q2_model.Status == GRB.OPTIMAL:
    q2_schedule = decode_schedule(q2_x, valid_trips)
    q2_schedule[["Date", "Home", "Visitor"]].to_csv(
        OUT_DIR / "q2_feasible_schedule_gurobi.csv", index=False)

    # Verify by re-running Q1 statistics on the new schedule
    q2_check = q1_stats(q2_schedule, teams)
    ok = all(
        q2_check[t]["home_dates"]  == summary[t]["home_dates"]  and
        q2_check[t]["away_dates"]  == summary[t]["away_dates"]  and
        q2_check[t]["home_counts"] == summary[t]["home_counts"] and
        q2_check[t]["away_counts"] == summary[t]["away_counts"]
        for t in teams
    )
    print("Status : Optimal")
    print(f"Q2 schedule passes all Q1 checks: {ok}")
    display(q2_schedule[["Date", "Home", "Visitor"]].head(12))

    # Excel export with the same per-team layout as Q1
    q2_export = q2_schedule.copy()
    q2_export["Date_only"] = pd.to_datetime(q2_export["Date"]).dt.normalize()
    write_team_matchups_excel(q2_export, ROOT / "team_matchups_by_opponent_gurobi.xlsx")
else:
    print(f"Unexpected status: {q2_model.Status}")


## Question 3 — Time-zone smoothness constraint

**Time zones**: ET = 0, CT = 1, MT = 2, PT = 3

**Constraint**: for every team $t$ and every window of 3 consecutive game-dates $(d_k, d_{k+1}, d_{k+2})$:
$$|\text{tz}(t,d_k)-\text{tz}(t,d_{k+1})| + |\text{tz}(t,d_{k+1})-\text{tz}(t,d_{k+2})| \le 3$$

**Linearisation**:
- $z_{d,t}$ : timezone of team $t$'s game on date $d$, linked to $x$ by a linear equality
- $\delta_{k,t} \ge |z_{d_k,t}-z_{d_{k+1},t}|$ via two one-sided inequalities
- Q3 constraint: $\delta_{k,t} + \delta_{k+1,t} \le 3$


In [ ]:
# Arena → time-zone index  (ET=0, CT=1, MT=2, PT=3)
TZ = {
    "TD Garden":                   0,   # Boston          ET
    "Madison Square Garden":       0,   # New York        ET
    "Barclays Center":             0,   # Brooklyn        ET
    "Wells Fargo Center":          0,   # Philadelphia    ET
    "Scotiabank Arena":            0,   # Toronto         ET
    "Kaseya Center":               0,   # Miami           ET
    "State Farm Arena":            0,   # Atlanta         ET
    "Rocket Mortgage FieldHouse":  0,   # Cleveland       ET
    "United Center":               1,   # Chicago         CT
    "Fiserv Forum":                1,   # Milwaukee       CT
    "Toyota Center":               1,   # Houston         CT
    "American Airlines Center":    1,   # Dallas          CT
    "Ball Arena":                  2,   # Denver          MT
    "Footprint Center":            2,   # Phoenix         MT
    "Crypto.com Arena":            3,   # Los Angeles     PT
    "Chase Center":                3,   # Golden State    PT
}
team_arena = games.groupby("Home")["Arena"].agg(lambda s: s.mode()[0]).to_dict()
team_tz    = {t: TZ[team_arena[t]] for t in teams}

print("Home arena time zones:")
for t, tz in sorted(team_tz.items(), key=lambda kv: kv[1]):
    print(f"  {['ET','CT','MT','PT'][tz]}  {t}")


In [ ]:
def check_q3_violations(sched, team_list, t_tz):
    """Return a DataFrame of 3-game windows that violate the Q3 constraint."""
    sched2 = sched.copy()
    sched2["Date"] = pd.to_datetime(sched2["Date"])
    rows = []
    for t in team_list:
        tg = sched2[(sched2["Home"] == t) | (sched2["Visitor"] == t)].copy()
        tg["tz"] = tg["Home"].map(t_tz)
        tg = tg.sort_values("Date").reset_index(drop=True)
        for k in range(len(tg) - 2):
            j1 = abs(int(tg.loc[k,   "tz"]) - int(tg.loc[k+1, "tz"]))
            j2 = abs(int(tg.loc[k+1, "tz"]) - int(tg.loc[k+2, "tz"]))
            if j1 + j2 >= 4:
                rows.append({
                    "Team":   t,
                    "Date1":  tg.loc[k,   "Date"].strftime("%Y-%m-%d"),
                    "Date2":  tg.loc[k+1, "Date"].strftime("%Y-%m-%d"),
                    "Date3":  tg.loc[k+2, "Date"].strftime("%Y-%m-%d"),
                    "Jump12": j1, "Jump23": j2, "Total": j1 + j2,
                })
    return pd.DataFrame(rows)


orig_viol = check_q3_violations(
    games[["DateStr", "Home", "Visitor"]].rename(columns={"DateStr": "Date"}),
    teams, team_tz)
print(f"Violations in original schedule: {len(orig_viol)}")
display(orig_viol)


In [ ]:
def build_q3_gurobi(home_date_set, away_date_set, h_count, a_count,
                    valid_trips, team_tz):
    """Build the Q3 model (Q2 constraints + timezone smoothness)."""
    m = gp.Model("NBA_Q3")
    m.setParam("OutputFlag", 0)

    x = m.addVars(valid_trips, vtype=GRB.BINARY, name="x")

    # (e)
    for i in teams:
        for d in home_date_set[i]:
            m.addConstr(
                gp.quicksum(x[d, i, j] for j in teams
                            if j != i and d in away_date_set[j]) == 1)
    # (f)
    for i in teams:
        for d in away_date_set[i]:
            m.addConstr(
                gp.quicksum(x[d, j, i] for j in teams
                            if j != i and d in home_date_set[j]) == 1)
    # (g)
    for i in teams:
        for j in teams:
            if i == j:
                continue
            m.addConstr(
                gp.quicksum(x[d, i, j] for d in dates
                            if d in home_date_set[i] and d in away_date_set[j])
                == h_count[(i, j)])
    # (h)
    for i in teams:
        for j in teams:
            if i == j:
                continue
            m.addConstr(
                gp.quicksum(x[d, j, i] for d in dates
                            if d in home_date_set[j] and d in away_date_set[i])
                == a_count[(i, j)])

    # z[d, t]: timezone index of team t on date d
    z = m.addVars(dates, teams, lb=0, ub=3, name="z")
    for d in dates:
        for t in teams:
            home_expr = team_tz[t] * gp.quicksum(
                x[d, t, j] for j in teams
                if j != t and d in home_date_set[t] and d in away_date_set[j])
            away_expr = gp.quicksum(
                team_tz[j] * x[d, j, t] for j in teams
                if j != t and d in home_date_set[j] and d in away_date_set[t])
            m.addConstr(z[d, t] == home_expr + away_expr, name=f"tz_{d}_{t}")

    # delta >= |z_k - z_{k+1}|
    delta = m.addVars(range(n - 1), teams, lb=0, ub=3, name="delta")
    for k in range(n - 1):
        d1, d2 = dates[k], dates[k + 1]
        for t in teams:
            m.addConstr(delta[k, t] >= z[d1, t] - z[d2, t], name=f"dpos_{k}_{t}")
            m.addConstr(delta[k, t] >= z[d2, t] - z[d1, t], name=f"dneg_{k}_{t}")

    # Q3 core constraint
    for k in range(n - 2):
        for t in teams:
            m.addConstr(delta[k, t] + delta[k + 1, t] <= 3, name=f"q3_{k}_{t}")

    m.setObjective(0, GRB.MINIMIZE)
    return m, x


q3_model, q3_x = build_q3_gurobi(
    home_date_set, away_date_set, h_count, a_count, valid_trips, team_tz)
print(f"Variables  : {q3_model.NumVars}")
print(f"Constraints: {q3_model.NumConstrs}")


In [ ]:
q3_model.optimize()

STATUS = {GRB.OPTIMAL: "Optimal", GRB.INFEASIBLE: "Infeasible"}
print(f"Q3 solver status: {STATUS.get(q3_model.Status, q3_model.Status)}")

if q3_model.Status == GRB.OPTIMAL:
    q3_schedule = decode_schedule(q3_x, valid_trips)
    q3_schedule[["Date", "Home", "Visitor"]].to_csv(
        OUT_DIR / "q3_feasible_schedule_gurobi.csv", index=False)
    viol = check_q3_violations(q3_schedule, teams, team_tz)
    print(f"Q3 violations in new schedule: {len(viol)}  (must be 0)")
    assert len(viol) == 0
    display(q3_schedule[["Date", "Home", "Visitor"]].head(12))

elif q3_model.Status == GRB.INFEASIBLE:
    print("No feasible schedule satisfying both Q2 and Q3 constraints exists.")
    q3_model.computeIIS()
    iis = [c.ConstrName for c in q3_model.getConstrs() if c.IISConstr]
    print(f"IIS contains {len(iis)} conflicting constraints.")
    print("First 10:", iis[:10])


### Q3 structural analysis — minimum unavoidable violations

Relaxed model: introduce binary slack $v_{k,t}\in\{0,1\}$ per window:
$$\delta_{k,t}+\delta_{k+1,t}\le 3+3\,v_{k,t}$$

**Objective**: minimise $\sum_{k,t}v_{k,t}$.  
If the optimum $> 0$, Q3 is structurally infeasible under any Q2-feasible schedule.


In [ ]:
def build_q3_relaxed_gurobi(home_date_set, away_date_set, h_count, a_count,
                             valid_trips, team_tz):
    """Relaxed Q3 model: minimise the number of violated 3-game windows."""
    m = gp.Model("NBA_Q3_Relaxed")
    m.setParam("OutputFlag", 0)

    x = m.addVars(valid_trips, vtype=GRB.BINARY, name="x")
    for i in teams:
        for d in home_date_set[i]:
            m.addConstr(
                gp.quicksum(x[d, i, j] for j in teams
                            if j != i and d in away_date_set[j]) == 1)
    for i in teams:
        for d in away_date_set[i]:
            m.addConstr(
                gp.quicksum(x[d, j, i] for j in teams
                            if j != i and d in home_date_set[j]) == 1)
    for i in teams:
        for j in teams:
            if i == j:
                continue
            m.addConstr(
                gp.quicksum(x[d, i, j] for d in dates
                            if d in home_date_set[i] and d in away_date_set[j])
                == h_count[(i, j)])
            m.addConstr(
                gp.quicksum(x[d, j, i] for d in dates
                            if d in home_date_set[j] and d in away_date_set[i])
                == a_count[(i, j)])

    z = m.addVars(dates, teams, lb=0, ub=3, name="z")
    for d in dates:
        for t in teams:
            home_expr = team_tz[t] * gp.quicksum(
                x[d, t, j] for j in teams
                if j != t and d in home_date_set[t] and d in away_date_set[j])
            away_expr = gp.quicksum(
                team_tz[j] * x[d, j, t] for j in teams
                if j != t and d in home_date_set[j] and d in away_date_set[t])
            m.addConstr(z[d, t] == home_expr + away_expr)

    delta = m.addVars(range(n - 1), teams, lb=0, ub=3, name="delta")
    for k in range(n - 1):
        d1, d2 = dates[k], dates[k + 1]
        for t in teams:
            m.addConstr(delta[k, t] >= z[d1, t] - z[d2, t])
            m.addConstr(delta[k, t] >= z[d2, t] - z[d1, t])

    # Binary slack: v[k,t]=1 allows window k of team t to violate Q3 (big-M = 3)
    v = m.addVars(range(n - 2), teams, vtype=GRB.BINARY, name="v")
    for k in range(n - 2):
        for t in teams:
            m.addConstr(delta[k, t] + delta[k + 1, t] <= 3 + 3 * v[k, t],
                        name=f"q3_relax_{k}_{t}")

    m.setObjective(
        gp.quicksum(v[k, t] for k in range(n - 2) for t in teams),
        GRB.MINIMIZE)
    return m, x, v


rl_model, rl_x, rl_v = build_q3_relaxed_gurobi(
    home_date_set, away_date_set, h_count, a_count, valid_trips, team_tz)
print(f"Variables  : {rl_model.NumVars}")
print(f"Constraints: {rl_model.NumConstrs}")
rl_model.optimize()

print(f"Relaxed status : {STATUS.get(rl_model.Status, rl_model.Status)}")
print(f"Minimum unavoidable Q3 violations: {int(round(rl_model.ObjVal))}")


In [ ]:
# Forced-violation windows + best (min-violation) schedule
forced = [
    {"Team": t, "Date1": dates[k], "Date2": dates[k+1],
     "Date3": dates[k+2], "WindowIndex": k}
    for k in range(n - 2)
    for t in teams
    if rl_v[k, t].X > 0.5
]
forced_df = pd.DataFrame(forced)
forced_df.to_csv(OUT_DIR / "q3_forced_violations_gurobi.csv", index=False)

mv_sched = decode_schedule(rl_x, valid_trips)
mv_sched[["Date", "Home", "Visitor"]].to_csv(
    OUT_DIR / "q3_min_violations_schedule_gurobi.csv", index=False)
mv_viol = check_q3_violations(mv_sched, teams, team_tz)

print(f"Forced-violation windows            : {len(forced_df)}")
print(f"Actual violations in best schedule  : {len(mv_viol)}")
print()
display(forced_df)
print("\nViolations per team:")
display(
    mv_viol.groupby("Team").size()
           .rename("Violations")
           .sort_values(ascending=False)
           .to_frame()
)


## Arena coordinates and distance matrix (scraper Part 2)

From a scraped BBR season (or the existing `nba_2024_schedule_scraped.csv`), take the core NBA home venues, **geocode** each with the [Google Geocoding API](https://developers.google.com/maps/documentation/geocoding/overview), and compute pairwise great-circle distances in **kilometers** with `geopy.distance.geodesic`.

**Install:** `pip install geopy`

**API key:** set `GOOGLE_MAPS_API_KEY` in the environment, or the cell will prompt once via `getpass`. If `nba_arena_coordinates.csv` and `nba_arena_distance_matrix_km.csv` already exist, the cell **reuses** them and skips geocoding.


In [ ]:
import getpass
import json as _json
import os
import time
import urllib.parse
import urllib.request

from geopy.distance import geodesic

ARENA_COORDS_CSV = ROOT / "nba_arena_coordinates.csv"
DIST_MATRIX_CSV  = ROOT / "nba_arena_distance_matrix_km.csv"
SCRAPED_CSV      = ROOT / "nba_2024_schedule_scraped.csv"


def _geocode(address: str, api_key: str) -> tuple[float, float]:
    params = urllib.parse.urlencode({"address": address, "key": api_key})
    url = f"https://maps.googleapis.com/maps/api/geocode/json?{params}"
    headers = {"User-Agent": "Mozilla/5.0"}
    req = urllib.request.Request(url, headers=headers, method="GET")
    with urllib.request.urlopen(req, timeout=30) as resp:
        payload = _json.loads(resp.read().decode("utf-8", errors="replace"))
    if payload.get("status") != "OK":
        raise RuntimeError(f"Geocode status={payload.get('status')!r} for {address!r}: {payload.get('error_message','')}")
    loc = payload["results"][0]["geometry"]["location"]
    return float(loc["lat"]), float(loc["lng"])


# Pick the source schedule (prefer scraped CSV; fall back to games.csv).
if SCRAPED_CSV.exists():
    _sched_src = pd.read_csv(SCRAPED_CSV)
else:
    _sched_src = games.copy()

_arena_clean = _sched_src["Arena"].astype(str).str.strip()
_sched_a     = _sched_src.assign(Arena=_arena_clean).loc[lambda d: d["Arena"].ne("") & d["Arena"].ne("nan")]
arena_counts = _sched_a.groupby("Arena").size()

MIN_GAMES_PER_ARENA = 0
arenas = sorted(a for a, c in arena_counts.items() if c >= MIN_GAMES_PER_ARENA)
print(f"Arenas in schedule: {len(arenas)}")

if ARENA_COORDS_CSV.exists() and DIST_MATRIX_CSV.exists():
    coords_df     = pd.read_csv(ARENA_COORDS_CSV)
    arena_coords  = {row["Arena"]: (float(row["lat"]), float(row["lon"])) for _, row in coords_df.iterrows()}
    dist_df       = pd.read_csv(DIST_MATRIX_CSV, index_col=0)
    dist_df.index   = dist_df.index.astype(str).str.strip()
    dist_df.columns = dist_df.columns.astype(str).str.strip()
    print(f"Reused existing {ARENA_COORDS_CSV.name} and {DIST_MATRIX_CSV.name}")
else:
    api_key = (os.environ.get("GOOGLE_MAPS_API_KEY") or "").strip()
    if not api_key:
        api_key = getpass.getpass("Google Maps Geocoding API key: ").strip()
    if not api_key:
        raise RuntimeError("A Google Geocoding API key is required.")

    arena_coords = {}
    for i, arena in enumerate(arenas):
        if i:
            time.sleep(0.25)
        clean = arena.split("(")[0].strip()  # strip "(IV)" suffixes etc.
        arena_coords[arena] = _geocode(f"{clean}, USA", api_key)
        lat, lon = arena_coords[arena]
        print(f"  {i+1:2}/{len(arenas)}  {arena[:50]:<50}  ({lat:.5f}, {lon:.5f})")

    coords_df = pd.DataFrame(
        [{"Arena": a, "lat": arena_coords[a][0], "lon": arena_coords[a][1]} for a in arenas]
    )
    coords_df.to_csv(ARENA_COORDS_CSV, index=False)
    print(f"Saved {ARENA_COORDS_CSV.resolve()}")

    n_a = len(arenas)
    km_mat = [[0.0] * n_a for _ in range(n_a)]
    for i in range(n_a):
        pi = arena_coords[arenas[i]]
        for j in range(i + 1, n_a):
            pj = arena_coords[arenas[j]]
            dkm = float(geodesic(pi, pj).meters) / 1000.0
            km_mat[i][j] = km_mat[j][i] = dkm
    dist_df = pd.DataFrame(km_mat, index=arenas, columns=arenas)
    dist_df.to_csv(DIST_MATRIX_CSV)
    print(f"Saved {DIST_MATRIX_CSV.resolve()}")

dist_df.iloc[:6, :6]


## Part 3 — Minimax travel IP on top of (e)–(h)

Pure integer program: rules **(e)** (home dates), **(f)** (away dates), **(g)** (home opponent counts $HG_{i,j}$), **(h)** (away opponent counts $AG_{i,j}$), plus **location / travel / minimax $M$**.

**Pre-computed (derived from `summary` so variables stay consistent with Q1–Q3):**
- $HD_i = \{d \in D : \alpha_{id}=1\}$, $AD_i = \{d \in D : \beta_{id}=1\}$
- $HG_{i,j} = b_{ij}$, $AG_{i,j} = c_{ij}$
- $\mathrm{Dist}_{a,b}$: km between the home arenas of teams $a$ and $b$ (rows/columns of `dist_df`).

**Variables:** $X_{i,j,d}$, $L_{i,d,a}$, $W_{i,d,a,b}$, $M \ge 0$.

**Extra linking:** if team $i$ has no game on date $d$, set $L_{i,d,i}=1$ (rest at home).

**Objective:** minimize $M$.

**Outputs:** `nba_part3_optimal_schedule.csv`, `team_matchups_by_opponent_part3_optimal.xlsx`, `nba_part3_optimal_team_travel_km.csv`.


In [ ]:
# Bridge from Q1/Q2 variables (`summary`, `teams`, `games`) to the matrix-style
# parameters (alpha, beta, b, c, D, nT, nD) the Part 3 IP expects — single source of truth.

D         = sorted(games["Date_only"].unique())
D_index   = {d: k for k, d in enumerate(D)}
nD, nT    = len(D), len(teams)
team_index = {t: k for k, t in enumerate(teams)}

alpha = np.zeros((nT, nD), dtype=np.int8)   # (a) i home on d
beta  = np.zeros((nT, nD), dtype=np.int8)   # (d) i away on d
b     = np.zeros((nT, nT), dtype=np.int16)  # (b) home opp counts
c     = np.zeros((nT, nT), dtype=np.int16)  # (c) away opp counts

for ti, i in enumerate(teams):
    for d_str in summary[i]["home_dates"]:
        alpha[ti, D_index[pd.Timestamp(d_str).normalize()]] = 1
    for d_str in summary[i]["away_dates"]:
        beta[ti,  D_index[pd.Timestamp(d_str).normalize()]] = 1
    for j, cnt in summary[i]["home_counts"].items():
        b[ti, team_index[j]] = cnt
    for j, cnt in summary[i]["away_counts"].items():
        c[ti, team_index[j]] = cnt

# Sanity checks
for ti, i in enumerate(teams):
    assert alpha[ti].sum() == b[ti].sum(), (i, "home games", alpha[ti].sum(), b[ti].sum())
    assert beta[ti].sum()  == c[ti].sum(), (i, "away games", beta[ti].sum(),  c[ti].sum())
assert b.sum() == c.sum(), (b.sum(), c.sum())

print(f"|T|={nT}, |D|={nD}, total scheduled games={int(b.sum())}")


In [ ]:
# Part 3 — (e)-(h) + location L, travel W, minimax M

# Map each team to its home arena, normalising to the rows of dist_df.
_ARENA_ALIASES: dict[str, str] = {
    "madison square garden":      "Madison Square Garden (IV)",
    "rocket mortgage fieldhouse": "Rocket Arena",
    "quicken loans arena":        "Rocket Arena",
}


def _matrix_arena(raw: str) -> str:
    import unicodedata
    s = unicodedata.normalize("NFKC", str(raw)).strip()
    if not s or s.lower() == "nan":
        return ""
    if s in dist_df.index:
        return s
    return _ARENA_ALIASES.get(s.casefold(), s)


home_arena: dict[str, str] = {}
for t in teams:
    ar = games.loc[games["Home"] == t, "Arena"].astype(str).str.strip()
    if ar.empty:
        raise RuntimeError(f"No home rows for team {t!r}.")
    home_arena[t] = _matrix_arena(str(ar.mode().iloc[0]))

missing = [(t, home_arena[t]) for t in teams if home_arena[t] not in dist_df.index]
if missing:
    raise RuntimeError(f"Arenas missing from distance matrix: {missing[:8]}")

km = np.zeros((nT, nT), dtype=float)
for ti, i in enumerate(teams):
    ai = home_arena[i]
    for tj, j in enumerate(teams):
        km[ti, tj] = float(dist_df.loc[ai, home_arena[j]])

GUROBI_PART3_TIMELIMIT_S = 0  # 0 = no time limit

model3 = gp.Model("nba_minimax_travel")
model3.Params.OutputFlag = 1
if GUROBI_PART3_TIMELIMIT_S > 0:
    model3.Params.TimeLimit = float(GUROBI_PART3_TIMELIMIT_S)

ij_dk = [(ti, tj, dk) for ti in range(nT) for tj in range(nT) for dk in range(nD) if ti != tj]
X3 = model3.addVars(ij_dk, vtype=GRB.BINARY, name="X")

# (e) home dates
for ti in range(nT):
    for dk in range(nD):
        model3.addConstr(
            gp.quicksum(X3[ti, tj, dk] for tj in range(nT) if tj != ti) == int(alpha[ti, dk]))

# (f) away dates
for ti in range(nT):
    for dk in range(nD):
        model3.addConstr(
            gp.quicksum(X3[tj, ti, dk] for tj in range(nT) if tj != ti) == int(beta[ti, dk]))

# (g) HG[i,j]
for ti in range(nT):
    for tj in range(nT):
        if ti == tj:
            continue
        model3.addConstr(gp.quicksum(X3[ti, tj, dk] for dk in range(nD)) == int(b[ti, tj]))

# (h) AG[i,j]
for ti in range(nT):
    for tj in range(nT):
        if ti == tj:
            continue
        model3.addConstr(gp.quicksum(X3[tj, ti, dk] for dk in range(nD)) == int(c[ti, tj]))

# L[i,d,a]: team i on calendar day d is at the home venue of team a
L3 = model3.addVars(nT, nD, nT, vtype=GRB.BINARY, name="L")
for ti in range(nT):
    for dk in range(nD):
        model3.addConstr(gp.quicksum(L3[ti, dk, a] for a in range(nT)) == 1)

# Match locations to X on game days
for ti in range(nT):
    for dk in range(nD):
        model3.addConstr(
            L3[ti, dk, ti] >= gp.quicksum(X3[ti, tj, dk] for tj in range(nT) if tj != ti))

for ti in range(nT):
    for tj in range(nT):
        if ti == tj:
            continue
        for dk in range(nD):
            model3.addConstr(L3[tj, dk, ti] >= X3[ti, tj, dk])

# Off-days: rest at home venue
for ti in range(nT):
    for dk in range(nD):
        if int(alpha[ti, dk]) == 0 and int(beta[ti, dk]) == 0:
            model3.addConstr(L3[ti, dk, ti] == 1)

# W[i,d,a,b]: move team i from site a (day d) to site b (day d+1)
W3 = model3.addVars(nT, nD - 1, nT, nT, vtype=GRB.BINARY, name="W")
for ti in range(nT):
    for dk in range(nD - 1):
        for ta in range(nT):
            for tb in range(nT):
                model3.addConstr(W3[ti, dk, ta, tb] <= L3[ti, dk, ta])
                model3.addConstr(W3[ti, dk, ta, tb] <= L3[ti, dk + 1, tb])
                model3.addConstr(W3[ti, dk, ta, tb] >= L3[ti, dk, ta] + L3[ti, dk + 1, tb] - 1)

M3 = model3.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="M")
for ti in range(nT):
    model3.addConstr(
        gp.quicksum(
            km[ta, tb] * W3[ti, dk, ta, tb]
            for dk in range(nD - 1)
            for ta in range(nT)
            for tb in range(nT)
        )
        <= M3)

model3.setObjective(M3, GRB.MINIMIZE)
model3.optimize()

_labels = {
    GRB.OPTIMAL: "OPTIMAL", GRB.TIME_LIMIT: "TIME_LIMIT",
    GRB.INFEASIBLE: "INFEASIBLE", GRB.INF_OR_UNBD: "INF_OR_UNBD",
    GRB.UNBOUNDED: "UNBOUNDED",
}
print(f"Part 3 Gurobi status: {_labels.get(model3.Status, model3.Status)}")

df_part3_opt: pd.DataFrame | None = None
df_part3_team_travel: pd.DataFrame | None = None
if model3.Status in (GRB.OPTIMAL, GRB.TIME_LIMIT):
    try:
        print(f"  Minimax M (km): {model3.ObjVal:,.3f}")
    except gp.GurobiError:
        print("  No objective value available.")
    if model3.Status == GRB.TIME_LIMIT and hasattr(model3, "ObjBound"):
        print(f"  Best bound: {model3.ObjBound:,.3f}  MIP gap: {getattr(model3, 'MIPGap', float('nan')) * 100:.4f}%")
elif model3.Status == GRB.INFEASIBLE:
    print("  Model infeasible — check data / linking constraints.")
else:
    print("  No incumbent — see Gurobi status above.")

# Export incumbent schedule
if getattr(model3, "SolCount", 0) > 0:
    games_p3: list[dict] = []
    for dk, d in enumerate(D):
        for ti, hi in enumerate(teams):
            for tj, vis in enumerate(teams):
                if ti == tj:
                    continue
                if X3[ti, tj, dk].X > 0.5:
                    games_p3.append({
                        "Date":        pd.Timestamp(d).normalize(),
                        "Home":        hi,
                        "Visitor":     vis,
                        "Arena":       home_arena[hi],
                        "MinimaxM_km": float(M3.X),
                    })
    df_part3_opt = (
        pd.DataFrame(games_p3)
        .sort_values(["Date", "Home", "Visitor"])
        .reset_index(drop=True)
    )
    df_part3_opt["Date_only"] = pd.to_datetime(df_part3_opt["Date"]).dt.normalize()

    out_p3_csv = ROOT / "nba_part3_optimal_schedule.csv"
    out_sched = df_part3_opt.drop(columns=["Date_only", "MinimaxM_km"], errors="ignore").copy()
    out_sched["Date"] = pd.to_datetime(out_sched["Date"]).dt.date
    out_sched.to_csv(out_p3_csv, index=False)
    print(f"Saved Part 3 optimal schedule ({len(df_part3_opt)} games) to {out_p3_csv.resolve()}")

    out_p3_xlsx = ROOT / "team_matchups_by_opponent_part3_optimal.xlsx"
    write_team_matchups_excel(df_part3_opt.drop(columns=["MinimaxM_km"], errors="ignore"), out_p3_xlsx)
    print(f"Saved per-team matchup sheets to {out_p3_xlsx.resolve()}")
    print(f"  (incumbent minimax M = {float(M3.X):,.3f} km)")

    rows_tr: list[dict] = []
    for ti in range(nT):
        tot = sum(
            km[ta, tb] * float(W3[ti, dk, ta, tb].X)
            for dk in range(nD - 1)
            for ta in range(nT)
            for tb in range(nT)
        )
        rows_tr.append({"Team": teams[ti], "TravelKM": tot})
    df_part3_team_travel = (
        pd.DataFrame(rows_tr).sort_values("TravelKM", ascending=False).reset_index(drop=True)
    )
    df_part3_team_travel.insert(0, "Rank", range(1, len(df_part3_team_travel) + 1))

    mx_w = float(df_part3_team_travel["TravelKM"].max())
    print(f"Per-team travel (from W): max = {mx_w:,.3f} km | mean = {df_part3_team_travel['TravelKM'].mean():,.3f} | "
          f"min = {df_part3_team_travel['TravelKM'].min():,.3f}")
    if abs(mx_w - float(M3.X)) > 1e-2:
        print(f"  (note: max team travel {mx_w:,.3f} vs M = {float(M3.X):,.3f} — expect equality up to numerical tolerance)")

    out_p3_travel = ROOT / "nba_part3_optimal_team_travel_km.csv"
    df_part3_team_travel.to_csv(out_p3_travel, index=False)
    print(f"Saved {out_p3_travel.resolve()}")

    display(out_sched.head(15))
    display(df_part3_team_travel)


## Part 4 — Distance traveled per team (from `games.csv`)

For each team, games are ordered by `Date`; each game's venue is the `Arena` column. Total travel is the sum of great-circle **km** between consecutive game arenas, looked up in `dist_df`.

If a venue string does not match a matrix row/column, an alias map (same MSG / Cleveland fixes as Part 3) is applied; any leg that still touches an unknown arena is **skipped** and counted in the printed warning.

Output: `games_csv_team_travel_km.csv` with columns `Team`, `TravelKM`, `Rank`.


In [ ]:
from collections import defaultdict


def _resolve_arena_p4(name: str, arena_index: set[str]) -> str:
    import unicodedata
    n = unicodedata.normalize("NFKC", str(name or "")).strip()
    if not n or n.lower() == "nan":
        return ""
    if n in arena_index:
        return n
    return _ARENA_ALIASES.get(n.casefold(), n)


def travel_km_per_team(
    sched: pd.DataFrame,
    dist_df: pd.DataFrame,
    team_list: list[str],
) -> tuple[dict[str, float], int, list[str]]:
    """Chronological sum of km between consecutive game arenas per team."""
    arena_index: set[str] = set(dist_df.index.astype(str))
    arena_seq: dict[str, list[tuple[pd.Timestamp, str]]] = defaultdict(list)
    unknown_venues: set[str] = set()

    for r in sched.sort_values("Date_only").itertuples(index=False):
        ar_raw = getattr(r, "Arena", "")
        ar = _resolve_arena_p4(str(ar_raw), arena_index)
        if not ar:
            continue
        if ar not in arena_index:
            unknown_venues.add(ar_raw if isinstance(ar_raw, str) else str(ar_raw))
        hm, vis = str(r.Home), str(r.Visitor)
        arena_seq[hm].append((r.Date_only, ar))
        arena_seq[vis].append((r.Date_only, ar))

    travel: dict[str, float] = {}
    skipped_legs = 0
    for t in team_list:
        seq = sorted(arena_seq[t])
        total = 0.0
        for k in range(len(seq) - 1):
            a1, a2 = seq[k][1], seq[k + 1][1]
            if a1 == a2:
                continue
            if a1 not in arena_index or a2 not in arena_index:
                skipped_legs += 1
                continue
            total += float(dist_df.loc[a1, a2])
        travel[t] = total

    return travel, skipped_legs, sorted(unknown_venues)


travel_g, n_skip, unknown = travel_km_per_team(games, dist_df, teams)
if unknown:
    print("Arena labels in games.csv not in matrix (add alias if needed):", unknown)
if n_skip:
    print(f"Skipped {n_skip} inter-game leg(s) (missing arena in matrix after aliases).")

travel_df = (
    pd.DataFrame([{"Team": t, "TravelKM": travel_g[t]} for t in teams])
    .sort_values("TravelKM", ascending=False)
    .reset_index(drop=True)
)
travel_df.insert(0, "Rank", range(1, len(travel_df) + 1))

mx, mu, mn = travel_df["TravelKM"].max(), travel_df["TravelKM"].mean(), travel_df["TravelKM"].min()
print(f"Max team travel:  {mx:,.1f} km")
print(f"Mean team travel: {mu:,.1f} km")
print(f"Min team travel:  {mn:,.1f} km")

OUT_TRAVEL = ROOT / "games_csv_team_travel_km.csv"
travel_df.to_csv(OUT_TRAVEL, index=False)
print(f"Saved {OUT_TRAVEL.resolve()}")
travel_df
